# Demo 06b — k-Nearest Neighbours

**Class 06 · Blocks 3-4** (slides: `slides/06_trees_knn.md`)

Five live demos, each with more experiments than the slides show:

* **K1 — KNN from scratch** with `jax.numpy`: pairwise distances, the $k$ nearest, the vote.
* **K2 — Scaling and $k$:** the same data with and without standardization; the U-shaped curve of $k$; weights.
* **K3 — The curse of dimensionality:** distances concentrate; irrelevant features ruin KNN.
* **K4 — KNN for regression** and its failure outside the data.
* **K5 — Cost, and all the learners of Classes 03-06 on one protocol.**

In [ ]:
import time
import warnings
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from scipy import stats
from sklearn.datasets import make_moons
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

jax.config.update("jax_enable_x64", True)
warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (9, 4)
pl.Config.set_tbl_rows(40)
pl.Config.set_fmt_str_lengths(60)
pl.Config.set_float_precision(3)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")
DATA = Path("../../datasets")


def table(rows, cols):
    keep = {"k"}
    rows = [tuple(f"{v:g}" if c in keep and isinstance(v, float) else v for c, v in zip(cols, r)) for r in rows]
    print(pl.DataFrame(rows, schema=cols, orient="row"))


def load_table(name, target):
    """Read `datasets/<name>.csv.zst` with polars: null -> median / "missing", text -> one-hot."""
    df = pl.read_csv(DATA / f"{name}.csv.zst")
    X = df.drop(target)
    X = X.with_columns([pl.col(c).fill_null(pl.col(c).median()) for c, t in X.schema.items() if t.is_numeric()])
    cats = [c for c, t in X.schema.items() if t == pl.String]
    if cats:
        X = X.with_columns([pl.col(c).fill_null("missing") for c in cats]).to_dummies(columns=cats)
    return X.to_numpy().astype(float), df[target].to_numpy(), X.columns


Xb, yb, names_b = load_table("breast_cancer", "target")
Xd, yd, _ = load_table("digits", "target")
yd = yd.astype(int)
Xg, cg, _ = load_table("german_credit", "class")
yg = (cg == "bad").astype(int)
Xw, qw, _ = load_table("wine_quality", "quality")
yw = (qw >= 6).astype(int)
Xh, nh, _ = load_table("heart_disease", "num")
yh = (nh == ">50_1").astype(int)
Xs, ys_, _ = load_table("spambase", "spam")
Xs = np.log1p(Xs)
ys_ = ys_.astype(int)
SETS = {"breast_cancer": (Xb, yb), "digits": (Xd / 16.0, yd), "spambase": (Xs, ys_), "wine (quality >= 6)": (Xw, yw), "german_credit": (Xg, yg), "heart_disease": (Xh, yh)}

## K1 — KNN from scratch

**The idea.** To classify a new point, find the $k$ training points closest to it and let them vote. There is **no
training**: the model *is* the training set ("lazy learning"). All the work is in the *distance*.

$$d(a, b) = \lVert a - b\rVert_2 = \sqrt{\sum_j (a_j - b_j)^2}$$

With the trick of Class 05, $\lVert a - b\rVert^2 = \lVert a\rVert^2 + \lVert b\rVert^2 - 2a^\top b$, all pairwise squared
distances between $m$ test points and $n$ training points are one matrix product.

In [ ]:
def pairwise_sq_dists(A, B):
    return jnp.sum(A**2, 1)[:, None] + jnp.sum(B**2, 1)[None, :] - 2 * A @ B.T


def knn_predict(Xtr, ytr, Xte, k=5, n_classes=2, weights="uniform"):
    d2 = jnp.maximum(pairwise_sq_dists(jnp.asarray(Xte), jnp.asarray(Xtr)), 0.0)
    neg, idx = jax.lax.top_k(-d2, k)                          # the k smallest distances
    votes = jax.nn.one_hot(jnp.asarray(ytr)[idx], n_classes)    # (m, k, K)
    if weights == "distance":
        w = 1.0 / (jnp.sqrt(-neg) + 1e-9)
        votes = votes * w[:, :, None]
    return np.asarray(votes.sum(1).argmax(1))


rows = []
for name, (X, y) in SETS.items():
    K = len(np.unique(y))
    a, b, ya, yb_ = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
    sc = StandardScaler().fit(a)
    a, b = sc.transform(a), sc.transform(b)
    t0 = time.time()
    ours = knn_predict(a, ya, b, k=5, n_classes=K)
    t_ours = time.time() - t0
    ref = KNeighborsClassifier(5).fit(a, ya)
    rows.append((name, float(np.mean(ours == yb_)), float(ref.score(b, yb_)), float(np.mean(ours == ref.predict(b))), t_ours))
table(rows, ["dataset (k=5, standardized)", "acc ours", "acc sklearn", "same pred.", "our time (s)"])

A tiny worked example (the one on the slides): 6 points in 2D, a query at $(2, 2)$, $k = 3$.

In [ ]:
P = np.array([[1.0, 1.0], [1.0, 3.0], [3.0, 1.0], [3.0, 3.0], [5.0, 5.0], [6.0, 4.0]])
labels = np.array([0, 0, 1, 1, 1, 0])
q = np.array([2.0, 2.0])
d = np.sqrt(((P - q) ** 2).sum(1))
order = np.argsort(d)
table([(i + 1, f"({P[i, 0]:g}, {P[i, 1]:g})", int(labels[i]), float(d[i])) for i in order], ["rank", "point", "class", "distance to (2, 2)"])
for k in (1, 3, 5):
    nn = order[:k]
    print(f"k = {k}: neighbours {(nn + 1).tolist()}, votes {np.bincount(labels[nn], minlength=2).tolist()} -> class {np.bincount(labels[nn], minlength=2).argmax()}")

**Other distances.** Euclidean ($p=2$), Manhattan ($p=1$), Chebyshev ($p=\infty$), cosine (angle), and Hamming (the number
of differing coordinates, for categorical data). The choice changes which points are neighbours:

In [ ]:
rows = []
for name in ("breast_cancer", "digits", "spambase", "heart_disease"):
    X, y = SETS[name]
    a, b, ya, yb_ = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
    sc = StandardScaler().fit(a)
    a, b = sc.transform(a), sc.transform(b)
    res = [name]
    for metric, kw in (("euclidean", {}), ("manhattan", {}), ("chebyshev", {}), ("cosine", {})):
        res.append(KNeighborsClassifier(5, metric=metric, algorithm="brute", **kw).fit(a, ya).score(b, yb_))
    rows.append(tuple(res))
table(rows, ["test accuracy, k=5", "euclidean", "manhattan", "chebyshev", "cosine"])

## K2 — Scaling and the choice of $k$

**Scaling.** The distance adds the squared differences of *all* features: a feature measured in large units decides the
neighbours. The standardization of Class 03 (training statistics only) is not optional.

In [ ]:
rows = []
for name, (X, y) in SETS.items():
    a, b, ya, yb_ = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
    raw = KNeighborsClassifier(5).fit(a, ya).score(b, yb_)
    sc = StandardScaler().fit(a)
    std = KNeighborsClassifier(5).fit(sc.transform(a), ya).score(sc.transform(b), yb_)
    spread = np.ptp(X, axis=0)
    rows.append((name, raw, std, float(spread.max() / max(spread[spread > 0].min(), 1e-12))))
table(rows, ["dataset", "acc, raw features", "acc, standardized", "range ratio max/min"])

**The choice of $k$** is the bias-variance trade-off of Class 01. $k = 1$: the boundary follows every point (low bias,
high variance, zero training error); a large $k$: a smooth boundary that approaches the majority class.

In [ ]:
Xm, ym = make_moons(300, noise=0.3, random_state=0)
Xm_te, ym_te = make_moons(3000, noise=0.3, random_state=1)
gx, gy = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))
fig, axes = plt.subplots(1, 5, figsize=(16, 3.2))
rows = []
for ax, k in zip(axes, (1, 5, 15, 51, 201)):
    m = KNeighborsClassifier(k).fit(Xm, ym)
    z = m.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    ax.contourf(gx, gy, z, alpha=0.15, cmap="coolwarm")
    ax.scatter(*Xm.T, c=ym, cmap="coolwarm", s=8)
    ax.set_title(f"k = {k}", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
    rows.append((k, m.score(Xm, ym), m.score(Xm_te, ym_te)))
plt.tight_layout()
plt.show()
table(rows, ["k", "train accuracy", "test accuracy"])

In [ ]:
ks = np.arange(1, 61, 2)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.4))
rows = []
for ax, name in zip(axes, ("breast_cancer", "digits", "german_credit")):
    X, y = SETS[name]
    accs = []
    for k in ks:
        pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(int(k)))
        accs.append(cross_val_score(pipe, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=0)).mean())
    ax.plot(ks, accs)
    ax.set(xlabel="k", title=name)
    best = int(np.argmax(accs))
    rows.append((name, int(ks[best]), accs[best], accs[0], accs[-1]))
axes[0].set_ylabel("5-fold CV accuracy")
plt.tight_layout()
plt.show()
table(rows, ["dataset", "best k", "CV acc at best k", "CV acc, k=1", "CV acc, k=59"])

**Weighting.** Instead of one vote per neighbour, weight each by $1/d$: close neighbours count more, ties become rare
and the dependence on $k$ weakens.

In [ ]:
rows = []
for name in ("breast_cancer", "digits", "spambase", "german_credit", "heart_disease"):
    X, y = SETS[name]
    res = [name]
    for k in (3, 15, 45):
        for w in ("uniform", "distance"):
            pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(k, weights=w))
            res.append(cross_val_score(pipe, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=0)).mean())
    rows.append(tuple(res))
table(rows, ["CV accuracy", "3 unif.", "3 dist.", "15 unif.", "15 dist.", "45 unif.", "45 dist."])

## K3 — The curse of dimensionality

In high dimension all points are almost **equally far** from each other: the "nearest" neighbour is hardly nearer than the
farthest, and a neighbourhood that holds a fixed share of the data is no longer *local* (Class 01).

In [ ]:
rng = np.random.default_rng(0)
rows = []
for d in (1, 2, 5, 10, 20, 50, 100, 500, 1000):
    Z = rng.uniform(size=(500, d))
    q = rng.uniform(size=(1, d))
    dist = np.sqrt(((Z - q) ** 2).sum(1))
    # edge length of a hypercube that holds 1% of uniform data
    rows.append((d, float(dist.min()), float(dist.max()), float(dist.min() / dist.max()), float(0.01 ** (1 / d))))
table(rows, ["dimension d", "nearest", "farthest", "nearest / farthest", "cube edge for 1% of the data"])

**Irrelevant features.** Add $r$ columns of pure noise to the breast-cancer data (standardized). Every learner is
trained and tested on the same split. KNN counts every column in the distance, so it degrades fastest; models that can
*weight* or *select* features resist.

In [ ]:
rng = np.random.default_rng(0)
a, b, ya, yb_ = train_test_split(Xb, yb, test_size=0.3, stratify=yb, random_state=0)
rows = []
for r in (0, 10, 50, 100, 300, 1000):
    A = np.c_[a, rng.normal(size=(len(a), r))]
    B = np.c_[b, rng.normal(size=(len(b), r))]
    sc = StandardScaler().fit(A)
    A, B = sc.transform(A), sc.transform(B)
    res = [r]
    for m in (KNeighborsClassifier(5), LogisticRegression(max_iter=2000), SVC(), DecisionTreeClassifier(min_samples_leaf=5, random_state=0), GaussianNB()):
        res.append(m.fit(A, ya).score(B, yb_))
    rows.append(tuple(res))
table(rows, ["noise features", "KNN k=5", "logistic", "SVM rbf", "tree", "naive Bayes"])

## K4 — KNN for regression

The prediction is the **mean** (or the distance-weighted mean) of the $k$ neighbours' targets: a local average. It is smooth
for large $k$ and follows the noise for $k=1$, and like a tree it cannot extrapolate.

In [ ]:
rng = np.random.default_rng(0)
xs = np.sort(rng.uniform(0, 6, 80))
ys = np.sin(xs) + rng.normal(0, 0.25, 80)
xg = np.linspace(-1, 7, 400)
fig, axes = plt.subplots(1, 4, figsize=(16, 3.2), sharey=True)
rows = []
for ax, k in zip(axes, (1, 3, 10, 40)):
    m = KNeighborsRegressor(k).fit(xs[:, None], ys)
    ax.scatter(xs, ys, s=8)
    ax.plot(xg, m.predict(xg[:, None]), color="tab:red")
    ax.plot(xg, np.sin(xg), "k:", lw=0.8)
    ax.set_title(f"k = {k}", fontsize=9)
    rows.append((k, float(np.sqrt(np.mean((m.predict(xs[:, None]) - ys) ** 2))), float(np.sqrt(np.mean((m.predict(xg[(xg > 0) & (xg < 6)][:, None]) - np.sin(xg[(xg > 0) & (xg < 6)])) ** 2)))))
plt.tight_layout()
plt.show()
table(rows, ["k", "train RMSE", "RMSE to the true curve (0 < x < 6)"])

In [ ]:
def load_california():
    X, y, names = load_table("california_housing", "MedHouseVal")
    for j, n in enumerate(names):
        if n in ("AveRooms", "AveBedrms", "Population", "AveOccup"):
            X[:, j] = np.log(X[:, j])
    return X, y.astype(float)


rows = []
for name, target in (("concrete", "strength"), ("abalone", "rings"), ("auto_mpg", "mpg"), ("california_housing", "MedHouseVal")):
    X, y = load_california() if name == "california_housing" else load_table(name, target)[:2]
    y = y.astype(float)
    if len(y) > 6000:
        idx = np.random.default_rng(0).choice(len(y), 6000, replace=False)
        X, y = X[idx], y[idx]
    bins = np.digitize(y, np.quantile(y, [0.25, 0.5, 0.75]))
    res = {}
    for label, mk in (("mean", None), ("linear", LinearRegression), ("knn k=1", lambda: KNeighborsRegressor(1)), ("knn k=5", lambda: KNeighborsRegressor(5)),
                      ("knn k=15", lambda: KNeighborsRegressor(15)), ("knn k=15 weighted", lambda: KNeighborsRegressor(15, weights="distance"))):
        errs = []
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=0).split(X, bins):
            sc = StandardScaler().fit(X[tr])
            if mk is None:
                pred = np.full(len(te), y[tr].mean())
            else:
                pred = mk().fit(sc.transform(X[tr]), y[tr]).predict(sc.transform(X[te]))
            errs.append(np.sqrt(np.mean((pred - y[te]) ** 2)))
        res[label] = float(np.mean(errs))
    rows.append((name, *res.values()))
table(rows, ["RMSE (5-fold)", "mean", "linear", "knn k=1", "knn k=5", "knn k=15", "knn k=15 weighted"])

## K5 — Cost, and all the learners on one protocol

**Cost.** Training is free (store the data); **prediction** compares the query with every training point:
$O(n\,d)$ per query. Spatial trees (**kd-tree**, **ball tree**) prune most of the comparisons in low dimension, and stop
helping when $d$ grows (the curse again).

In [ ]:
rng = np.random.default_rng(0)
rows = []
for n, d in ((2000, 2), (20000, 2), (20000, 10), (20000, 30), (20000, 100)):
    Xtr_ = rng.normal(size=(n, d))
    ytr_ = (Xtr_[:, 0] > 0).astype(int)
    Xq = rng.normal(size=(500, d))
    res = [n, d]
    for algo in ("brute", "kd_tree", "ball_tree"):
        m = KNeighborsClassifier(5, algorithm=algo)
        t0 = time.time()
        m.fit(Xtr_, ytr_)
        t_fit = time.time() - t0
        t0 = time.time()
        m.predict(Xq)
        res.append(time.time() - t0)
    rows.append(tuple(res))
table(rows, ["n", "d", "brute (s / 500 queries)", "kd-tree", "ball tree"])
print(f"memory of the model: n x d numbers; 1 000 000 examples x 100 features in float64 = {1_000_000 * 100 * 8 / 1e9:.1f} GB")

**One protocol, six learners.** Logistic regression (Class 03), Gaussian naive Bayes (Class 04), RBF SVM (Class 05), a pruned
decision tree and KNN (this class), on 5 stratified 75/25 splits with the standardization fitted on the training part;
$k$ and the tree's `min_samples_leaf` chosen by an inner 5-fold cross-validation.

In [ ]:
from sklearn.model_selection import GridSearchCV

learners = {
    "logistic": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
    "naive Bayes": lambda: make_pipeline(StandardScaler(), GaussianNB()),
    "SVM rbf": lambda: make_pipeline(StandardScaler(), SVC()),
    "tree": lambda: GridSearchCV(DecisionTreeClassifier(random_state=0), {"min_samples_leaf": [1, 5, 10, 20, 50]}, cv=5),
    "KNN": lambda: GridSearchCV(make_pipeline(StandardScaler(), KNeighborsClassifier()), {"kneighborsclassifier__n_neighbors": [1, 3, 5, 9, 15, 25]}, cv=5),
}
results = {}
timing = {}
for name, (X, y) in SETS.items():
    for rep in range(5):
        a, b, ya, yb_ = train_test_split(X, y, test_size=0.25, stratify=y, random_state=rep)
        for lname, mk in learners.items():
            t0 = time.time()
            m = mk().fit(a, ya)
            timing.setdefault((name, lname), []).append(time.time() - t0)
            results.setdefault((name, lname), []).append(m.score(b, yb_))
rows = []
for name in SETS:
    row = [name]
    for lname in learners:
        row.append(float(np.median(results[(name, lname)])))
    rows.append(tuple(row))
table(rows, ["median test accuracy", *learners])
print()
table([(lname, float(np.median([np.median(timing[(n, lname)]) for n in SETS]))) for lname in learners], ["learner", "median fit time incl. tuning (s)"])

**Is the difference real?** The corrected resampled $t$-test of Class 05 for KNN against the tree and the SVM:

In [ ]:
def corrected_ttest(a, b, ratio=1 / 3):
    d = np.asarray(a) - np.asarray(b)
    var = d.var(ddof=1)
    if var == 0:
        return 0.0, 1.0
    t = d.mean() / np.sqrt((1 / len(d) + ratio) * var)
    return float(t), float(2 * stats.t.sf(abs(t), len(d) - 1))


rows = []
for name in SETS:
    for A, B in (("KNN", "tree"), ("KNN", "SVM rbf"), ("KNN", "logistic")):
        t, p = corrected_ttest(results[(name, A)], results[(name, B)])
        rows.append((name, f"{A} - {B}", float(np.mean(results[(name, A)]) - np.mean(results[(name, B)])), t, p))
table(rows, ["dataset", "comparison", "mean difference", "t", "p"])

## Summary

* KNN stores the data and answers with the **$k$ nearest** points: no training, a nonlinear boundary, an honest
  "like the ones I know".
* Everything hangs on the **distance**: standardize first; irrelevant features and high dimension destroy it.
* $k$ is the bias-variance knob: $k = 1$ memorizes; a large $k$ smooths; choose it on validation data.
* Prediction is $O(nd)$ per query; kd-trees help only in low dimension.
* No learner wins everywhere: compare on the same protocol with a corrected test.